# INTI-3K · Notebook reproducible del envío final (`submission_notebook_vn.ipynb`)

**Hackathon IA PUCP 2026 · Equipo PASSWORD**

Corrido de principio a fin **solo con la data oficial** (`train.npz`, `test.npz`, `static.npz`, `sample_submission.csv`):

1. **Entrena cada miembro del ensamble final** (lista `MEMBERS`: arquitectura, semilla, épocas, `fno_modes`, variante de lags) con el 100 % del train y épocas fijas: `kaggle_pipeline/src/run_cv.py --refit` (mismo código que la CV; aquí se **importa**, no se copia).
2. **Exporta un único modelo** `EnsembleVN` (`kaggle_pipeline/src/export_ensemble.py`): `models/modelo.pt` + `models/norm_stats.npz` + `models/model_card.json`, con los pesos de mezcla por variable `BLEND` (salida de `blend.py` o `'uniform'`).
3. **Predice el test con ese modelo exportado** (recargado desde disco, igual que el producto `app/inti_inference.py`), verifica que coincide con el blend de las predicciones de cada refit, valida y escribe `submission.csv`.

Así la submission y el producto usan **exactamente los mismos pesos** (regla de las bases).

| Flag | Efecto |
|---|---|
| `TRAIN = True` | entrena todos los refits + exporta + submission (regenera `models/`) |
| `TRAIN = False` | NO entrena: carga `models/modelo.pt` y solo predice |
| `SMOKE = True` | prueba de humo: 2 miembros chicos, 1 época, CPU, train submuestreado; escribe TODO en una carpeta temporal (no toca `models/` ni `submission.csv`) |

**Rutas.** Data: `PASSWORD_DATA_DIR` (en Kaggle se detecta `/kaggle/input/...`), o `<repo>/data`. Salidas intermedias: `PASSWORD_OUT_DIR` (Kaggle: `/kaggle/working`). Código: el notebook busca `kaggle_pipeline/src` en `PASSWORD_REPO_DIR`, `..`, `.` o en `/kaggle/input/**` (subir el repo como dataset o `git clone` en `/kaggle/working`).

**Determinismo.** Semilla fija por miembro (`seed_everything(seed)` dentro de `run_cv`), `cudnn.deterministic=True`, `benchmark=False`, sin TF32, `use_deterministic_algorithms(True, warn_only=True)`.


In [ ]:
# ============================ CONFIGURACIÓN ============================
# TRAIN = True  -> entrena cada refit de MEMBERS, exporta EnsembleVN y genera submission.csv
# TRAIN = False -> carga models/modelo.pt + models/norm_stats.npz (arch='ensemble_vn') y solo predice
TRAIN = True
# SMOKE = True  -> prueba de humo (2 miembros chicos, 1 época, CPU) en carpeta temporal
SMOKE = False

# Miembros del envío final. Cada uno = un `run_cv.py --refit`. Claves:
#   model: convnext | multiscale | unet | fno     seed: semilla      epochs: épocas fijas (100 % train)
#   fno_modes: "m1,m2" (solo fno) o None          lags: "default" (±24 h) | "past" (causal)
#   tag: sufijo del nombre del miembro (p. ej. "m86")   extra: otros flags de run_cv (["--loss-w-uv", "1.3"], ...)
# Todos deben compartir el mismo `lags` (EnsembleVN comparte features y normalización: export lo verifica).
# >>> REEMPLAZAR por la lista final elegida en la Fase 3 (mismos flags que los refits evaluados en CV). <<<
MEMBERS = [
    dict(model="convnext",   seed=42,  epochs=24, fno_modes=None, lags="default", tag="", extra=[]),
    dict(model="multiscale", seed=102, epochs=24, fno_modes=None, lags="default", tag="", extra=[]),
    dict(model="unet",       seed=203, epochs=24, fno_modes=None, lags="default", tag="", extra=[]),
    dict(model="fno",        seed=304, epochs=24, fno_modes=None, lags="default", tag="", extra=[]),
]
# Pesos de mezcla por variable: "uniform" | ruta a un json de blend.py | dict {"members": [...], "weights": [[RH,T,U,V], ...]}
BLEND = "uniform"
BLEND_KEY = None          # shrink del json de blend.py (None = el de mejor cv_nested)
ID_EXP = "vn_final"       # id del experimento (nombre de la submission y model_card)
CV_MEDIA = None           # cv_media del ensamble (registro / nombre del archivo), p. ej. 0.4123
LB = None                 # LB público, si ya se conoce

SMOKE_MEMBERS = [         # misma semilla -> mismo submuestreo -> mismas stats (requisito del export)
    dict(model="convnext", seed=7, epochs=1, fno_modes=None, lags="default", tag="", extra=["--width", "16"]),
    dict(model="fno",      seed=7, epochs=1, fno_modes="6,4", lags="default", tag="", extra=["--width", "16"]),
]


In [ ]:
import glob
import json
import os
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ---- Determinismo (igual que submission_notebook.ipynb). CUBLAS_WORKSPACE_CONFIG antes de la 1.ª op CUDA.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)  # grid_sample backward CUDA no es determinista (solo avisa)

# ---- Código del repo (kaggle_pipeline/src): se importa, no se duplica
def find_repo() -> Path:
    cands = [os.environ.get("PASSWORD_REPO_DIR"), "..", ".", "/kaggle/working/repo"]
    cands += [str(Path(p).parents[1]) for p in glob.glob("/kaggle/input/**/kaggle_pipeline/src/run_cv.py", recursive=True)]
    for c in cands:
        if c and (Path(c) / "kaggle_pipeline" / "src" / "run_cv.py").exists():
            return Path(c).resolve()
    raise FileNotFoundError("No encuentro kaggle_pipeline/src: define PASSWORD_REPO_DIR o sube el repo como dataset")

ROOT_DIR = find_repo()
SRC_DIR = ROOT_DIR / "kaggle_pipeline" / "src"
for p in (SRC_DIR, ROOT_DIR / "app"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ---- Data oficial: PASSWORD_DATA_DIR > /kaggle/input > <repo>/data (paths.py busca en subcarpetas)
if not os.environ.get("PASSWORD_DATA_DIR"):
    hits = glob.glob("/kaggle/input/**/train.npz", recursive=True)
    if hits:
        os.environ["PASSWORD_DATA_DIR"] = str(Path(hits[0]).parent)
from paths import DATA_DIR, data_file  # noqa: E402

KAGGLE = Path("/kaggle/working").is_dir()
if SMOKE:
    WORK_DIR = Path(tempfile.mkdtemp(prefix="nb_vn_smoke_"))   # nada fuera de la carpeta temporal
    OUT_DIR, MODELS_OUT, SUB_DIR = WORK_DIR / "exp", WORK_DIR / "models", WORK_DIR / "submissions"
    SUBMISSION_PATH = WORK_DIR / "submission.csv"
    MEMBERS_RUN = SMOKE_MEMBERS
else:
    OUT_DIR = Path(os.environ.get("PASSWORD_OUT_DIR") or ("/kaggle/working" if KAGGLE else ROOT_DIR / "kaggle_pipeline" / "experiments" / "notebook_vn"))
    MODELS_OUT = Path("/kaggle/working/models") if KAGGLE else ROOT_DIR / "models"
    SUB_DIR = Path("/kaggle/working/submissions") if KAGGLE else ROOT_DIR / "submissions"
    SUBMISSION_PATH = Path("/kaggle/working/submission.csv") if KAGGLE else ROOT_DIR / "submission.csv"
    MEMBERS_RUN = MEMBERS
if not TRAIN and not SMOKE:
    MODELS_OUT = ROOT_DIR / "models"   # TRAIN=False: pesos versionados del repo (solo lectura)
REFIT_DIR = OUT_DIR / "refit"
for d in (OUT_DIR, MODELS_OUT, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)

import export_ensemble  # noqa: E402
import run_cv  # noqa: E402
from check_submission import build_submission, check_submission  # noqa: E402
from ensemble_vn import load_ensemble_vn  # noqa: E402

device = "cpu" if SMOKE or not torch.cuda.is_available() else "cuda"
print(f"repo={ROOT_DIR} | data={DATA_DIR} | out={OUT_DIR} | models={MODELS_OUT} | submission={SUBMISSION_PATH}")
print(f"device={device} | torch {torch.__version__} | TRAIN={TRAIN} | SMOKE={SMOKE} | {len(MEMBERS_RUN)} miembros")

te = np.load(data_file("test.npz"))
Xte, tte = te["X"], te["times"].astype(str)
static = dict(np.load(data_file("static.npz")))
sample_path = data_file("sample_submission.csv")
sample = pd.read_csv(sample_path)
print(f"Test X: {Xte.shape} | {tte[0]} -> {tte[-1]} | sample_submission: {sample.shape}")


In [ ]:
def member_argv(m: dict) -> list:
    """Flags de run_cv.py --refit para un miembro de MEMBERS."""
    a = ["--model", m["model"], "--seed", str(m["seed"]), "--epochs", str(m["epochs"]), "--refit",
         "--lags", m.get("lags", "default"), "--out-dir", str(OUT_DIR), "--data-dir", str(DATA_DIR)]
    if m.get("fno_modes"):
        a += ["--fno-modes", str(m["fno_modes"])]
    if m.get("tag"):
        a += ["--tag", m["tag"]]
    if SMOKE:
        a += ["--smoke", "--smoke-real", "--no-registro"]
    return a + list(m.get("extra", []))

def member_name(m: dict) -> str:
    """'{miembro}_{seed}' con la MISMA regla de nombres que run_cv (sufijos -tag / -lagpast)."""
    args = run_cv.build_parser().parse_args(member_argv(m))
    hp = run_cv._resolve_hparams(args)
    return f"{hp['member']}_{hp['seed']}"

NAMES = [member_name(m) for m in MEMBERS_RUN]
assert len(set(NAMES)) == len(NAMES), f"miembros repetidos: {NAMES}"
assert len({m.get('lags', 'default') for m in MEMBERS_RUN}) == 1, "todos los miembros deben usar el mismo lag_set"
print("Miembros:", NAMES)


## 1 · Refit de cada miembro (100 % del train)

In [ ]:
# Fase 1: un refit por miembro (100 % del train, épocas fijas, estado EMA final, sin mirar validación).
# Cada refit guarda en REFIT_DIR: model_{name}.pt, norm_{name}.npz, pred_test_{name}.npy.
if TRAIN:
    t_all = time.time()
    for m, nm in zip(MEMBERS_RUN, NAMES):
        t0 = time.time()
        print("=" * 90, f"\nREFIT {nm}: run_cv.py " + " ".join(member_argv(m)))
        run_cv.main(member_argv(m))
        print(f"REFIT {nm} listo en {(time.time() - t0) / 60:.1f} min")
    print(f"Todos los refits en {(time.time() - t_all) / 60:.1f} min")
else:
    print("TRAIN=False -> no se entrena (se usa models/modelo.pt)")


## 2 · Export de `EnsembleVN` y submission predicha con el modelo exportado

In [ ]:
# Fase 2: exportar EnsembleVN (modelo.pt + norm_stats.npz + model_card.json) y predecir el test CON EL
# MODELO EXPORTADO (recargado desde disco). export verifica que coincide con el blend de los pred_test_*.npy.
if TRAIN:
    blend_arg = BLEND
    if isinstance(BLEND, dict):
        blend_arg = str(OUT_DIR / "blend_notebook.json")
        with open(blend_arg, "w", encoding="utf-8") as f:
            json.dump(BLEND, f)
    res = export_ensemble.export(
        NAMES, blend_arg, REFIT_DIR, MODELS_OUT, sub_dir=SUB_DIR, data_dir=DATA_DIR, id_exp=ID_EXP + ("_SMOKE" if SMOKE else ""),
        cv_media=CV_MEDIA, lb=LB, blend_key=BLEND_KEY, device=device, submission_copy=SUBMISSION_PATH,
    )
    print("Submission:", res["submission"], "| max|diff| vs blend de refits:", res["diff"])
else:
    # Solo inferencia con los pesos versionados (mismas funciones que export_ensemble)
    model, stats = load_ensemble_vn(MODELS_OUT, static, device)
    need = set(sample["id"].str.slice(0, 10).unique())
    idx = np.array([i for i, t in enumerate(tte) if t in need], dtype=np.int64)
    t0 = time.time()
    pred = export_ensemble.predict_test(model, stats, Xte, tte, idx, device)
    print(f"Predicción de {len(idx)} horas en {time.time() - t0:.1f}s ({device})")
    sub = build_submission(pred, tte[idx], sample)
    sub.to_csv(SUBMISSION_PATH, index=False)


## 3 · Validación y paridad producto ↔ submission

In [ ]:
# Validación final de submission.csv + ficha del modelo + paridad con el producto (app/inti_inference.py)
assert check_submission(SUBMISSION_PATH, sample_path), "submission.csv inválido"
card_path = MODELS_OUT / "model_card.json"
if card_path.exists():
    card = json.loads(card_path.read_text(encoding="utf-8"))
    print(f"{card.get('name')} | {card.get('total_params', 0):,} params | pesos {card['weights']['size_mb']} MB "
          f"({card['weights'].get('dtype')}) | sha256 {card['weights']['sha256']}")

os.environ["INTI_MODELS_DIR"] = str(MODELS_OUT)   # el producto lee la MISMA carpeta de pesos
import inti_inference  # noqa: E402
inti_inference.reset_runner()
Y24 = inti_inference.predict_sequence(Xte, tte, static, out_idx=np.arange(24))
sub = pd.read_csv(SUBMISSION_PATH)
parts = sub["id"].str.split("_", expand=True)
pos = {t: i for i, t in enumerate(tte[:24])}
ti = parts[0].map(pos)
msk = ti.notna().values
ti, rr, cc = ti[msk].astype(int).values, parts[1][msk].astype(int).values, parts[2][msk].astype(int).values
par = {v: float(np.abs(Y24[ti, rr, cc, k] - sub.loc[msk, v].values).max()) for k, v in enumerate(["RH", "T", "U", "V"])}
print(f"Paridad producto vs submission (24 h): max|diff| = {par}")
assert max(par.values()) < 1e-3 + 5e-4, "PARIDAD FALLIDA"
sub.head()
